**Resetting of the stream**


In [0]:
%sql
DROP TABLE IF EXISTS retail_catalog.bronze.inventory_events;
DROP TABLE IF EXISTS retail_catalog.bronze.sku_master;
DROP TABLE IF EXISTS retail_catalog.bronze.warehouse_master;

In [0]:
for s in spark.streams.active:
    s.stop()

dbutils.fs.rm("s3://retail-ims/checkpoints/inventory", True)
dbutils.fs.rm("s3://retail-ims/schema/inventory", True)

dbutils.fs.rm("s3://retail-ims/schema/sku", True)
dbutils.fs.rm("s3://retail-ims/checkpoints/sku", True)

dbutils.fs.rm("s3://retail-ims/schema/warehouse", True)
dbutils.fs.rm("s3://retail-ims/checkpoints/warehouse", True)

True

**Start With Ingestion**

In [0]:
%sql
USE CATALOG retail_catalog;
USE SCHEMA bronze;

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit
from pyspark.sql.types import *

**Loading Files and Adding Metadata**

In [0]:
from pyspark.sql.functions import current_timestamp

def bronze_ingestion(source_path, table_name, schema_loc, checkpoint_loc, file_format="csv"):

    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", file_format) \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaLocation", schema_loc) \
        .load(source_path)

    # Add metadata
    df = df.withColumn("ingestion_timestamp", current_timestamp())

    return df.writeStream \
        .format("delta") \
        .option("checkpointLocation", checkpoint_loc) \
        .outputMode("append") \
        .trigger(availableNow=True) \
        .toTable(table_name)    

**Ingestion of inventory table**

In [0]:
bronze_ingestion(
    source_path="s3://retail-ims/inventory_events/",
    table_name="retail_catalog.bronze.inventory_events",
    schema_loc="s3://retail-ims/schema/inventory",
    checkpoint_loc="s3://retail-ims/checkpoints/inventory"
)

In [0]:
bronze_ingestion(
    source_path="s3://retail-ims/sku_master/",
    table_name="retail_catalog.bronze.sku_master",
    schema_loc="s3://retail-ims/schema/sku",
    checkpoint_loc="s3://retail-ims/checkpoints/sku"
)

In [0]:
bronze_ingestion(
    source_path="s3://retail-ims/warehouse_master/",
    table_name="retail_catalog.bronze.warehouse_master",
    schema_loc="s3://retail-ims/schema/warehouse",
    checkpoint_loc="s3://retail-ims/checkpoints/warehouse"
)

In [0]:
%sql
SELECT COUNT(*) FROM retail_catalog.bronze.inventory_events ;


COUNT(*)
301500


In [0]:
%sql
SELECT COUNT(*) FROM retail_catalog.bronze.sku_master;

COUNT(*)
1000


In [0]:
%sql
SELECT * FROM retail_catalog.bronze.warehouse_master;

warehouse_id,location,capacity,_rescued_data,ingestion_timestamp
WH-01,Mumbai,20000,null,2026-05-04T18:11:25.267Z
WH-02,Bangalore,10000,null,2026-05-04T18:11:25.267Z
WH-03,Hyderabad,12500,null,2026-05-04T18:11:25.267Z
WH-04,Chennai,8000,null,2026-05-04T18:11:25.267Z
WH-05,Delhi,15500,null,2026-05-04T18:11:25.267Z
